# Feature selection by category: Claims

Finds the strongest **claims** features (HCP monthly claim aggregates, NB01-NB06), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with **all 28 cohorts** (c1-c28). The notebook reads only this category's source tables and fills empty values with the same rules as `NBA_3_1_Combine_Physician_Data`.

**Part 1: data set (same in every category notebook, NBA 3.1 method)**
1. Writer definition: an HCP-month is an existing writer if `TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0 (empty = 0). Every other HCP-month is a non-writer.
2. Two modeling data sets: writer HCP-months and non-writer HCP-months.
3. Mean / median analysis: weighted mean and median of every feature for writers and for non-writers. Features are ranked by how well they separate the two groups (`DISCRIMINATION` = \|2 x AUC - 1\|: 0 = same values in both groups, 1 = no overlap).

**Part 2: category model (per group)**

4. Statistical clean-up as in NBA 3.1: removal stages 1 constant, 2 near-constant, 3 low fill, 4 near-copy; flags 5 no signal and 6 unstable (flags only). It runs before the model so the model sees only usable features.
5. One deliberately over-fitted LightGBM Poisson model per group on all 28 cohorts (no hold-out), target = Kerendia NBRx in the prediction month, weighted by the sample group weight.
6. SHAP values (LightGBM `pred_contrib`, exact tree SHAP) on every row; features ranked by weighted mean \|SHAP\|.
7. Best features per group: SHAP rank with the mean / median and clean-up results side by side.

**Category config** (cell 3) is the only cell that differs between the category notebooks; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4), so every category uses exactly the same code. **Widgets:** `RUN_NAME`, `SAMPLE_LIST` (empty = full HCP universe), `TEST_HCPS` / `MAX_FEATURES` (0 = all; set both for a quick test), model settings, `TOP_N`. Outputs go to `hcp_level_model/dev_pipeline/notebook_output/feature_selection_by_category/<RUN_NAME>/<category>/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "claims"
CATEGORY_LABEL = "Claims: HCP monthly claim aggregates (NB01-NB06)"
ADD_CALENDAR   = False   # CALENDAR_MONTH, CALENDAR_YEAR, MONTHS_SINCE_KER_LAUNCH (static notebook only)

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"
CLAIMS_RUN = DEV + "full_2026_11_13/hcp_claims/"          # latest full claims run, as in Combine

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

SOURCE_TABLES = [
    _t("provider_claim_rx_monthly_cohort", "Rx claims", CLAIMS_RUN + "provider_claim_rx_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_lab_monthly_cohort", "Lab claims", CLAIMS_RUN + "provider_claim_lab_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_diagnosis_monthly_cohort", "Dx claims", CLAIMS_RUN + "provider_claim_diagnosis_monthly_cohort", "delta", "COHORT_ID"),
    # NB04 repeats 5 Rx totals of NB01: kept from NB01 (same as Combine)
    _t("final_claims_monthly_cohort", "Monthly claims", CLAIMS_RUN + "final_claims_monthly_cohort", "delta", "COHORT_ID",
       drop_cols=["RUNNING_TOTAL_RX_CLAIMS", "TOTAL_RX_CLAIMS_IN_LAST_1_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_3_MONTH",
                  "TOTAL_RX_CLAIMS_IN_LAST_6_MONTH", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH"]),
    _t("claim_rx_nps_monthly_cohort", "Rx NPS claims", CLAIMS_RUN + "claim_rx_nps_monthly_cohort", "delta", "COHORT_ID"),
    _t("provider_claim_eligible_patient_cohort", "Eligible patients", CLAIMS_RUN + "provider_claim_eligible_patient_cohort", "delta", "COHORT_ID"),
]

In [0]:
%run ./00_category_feature_selection_core